# Проверка RuBERT: оригинальные данные vs очищенная аугментация

Train 1: `Data/train_after_eda.csv`  
Train 2: `Data/data_after_stage3gpt.csv`  
Test: `Data/data_test.csv`  
Модели и настройки совпадают с `notebooks/augmentation.ipynb`: `rubert-tiny2` и `rubert-base`, 15 эпох.

In [ ]:
import gc
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import set_seed

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification.rubert_classifier import train_and_evaluate
from src.utils.data_loader import TEXT_COL, LABEL_COL

DATA_DIR = PROJECT_ROOT / "Data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

SEED = 42
ORIGINAL_TRAIN_PATH = DATA_DIR / "train_after_eda.csv"
GPT_TRAIN_PATH = DATA_DIR / "data_after_stage3gpt.csv"
TEST_PATH = DATA_DIR / "data_test.csv"

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

reset_seed()

In [ ]:
train_datasets = [
    {
        "dataset": "original",
        "path": ORIGINAL_TRAIN_PATH,
        "df": pd.read_csv(ORIGINAL_TRAIN_PATH),
    },
    {
        "dataset": "stage3gpt",
        "path": GPT_TRAIN_PATH,
        "df": pd.read_csv(GPT_TRAIN_PATH),
    },
]
df_test = pd.read_csv(TEST_PATH)

print(f"Test:  {TEST_PATH} | {len(df_test)} rows | {df_test[LABEL_COL].nunique()} classes")

for item in train_datasets:
    df_train_i = item["df"]
    print(f"Train [{item['dataset']}]: {item['path']} | {len(df_train_i)} rows | {df_train_i[LABEL_COL].nunique()} classes")
    missing_in_train = sorted(set(df_test[LABEL_COL]) - set(df_train_i[LABEL_COL]))
    if missing_in_train:
        raise ValueError(f"В train [{item['dataset']}] отсутствуют классы из test: {missing_in_train}")

for item in train_datasets:
    dist = item["df"][LABEL_COL].value_counts().sort_values()
    print(f"\n[{item['dataset']}] распределение классов")
    display(dist.to_frame("train_count").head(15))
    display(dist.describe().to_frame("train_count_stats"))

In [ ]:
RUBERT_CONFIGS = [
    {
        "model_name": "cointegrated/rubert-tiny2",
        "short_name": "rubert-tiny2",
        "lr": 5e-4,
        "num_epochs": 15,
        "batch_size": 32,
    },
    {
        "model_name": "DeepPavlov/rubert-base-cased",
        "short_name": "rubert-base",
        "lr": 5e-5,
        "num_epochs": 15,
        "batch_size": 32,
    },
]

RUBERT_CONFIGS

In [ ]:
rubert_comparison_results = []

for train_item in train_datasets:
    dataset_name = train_item["dataset"]
    train_path = train_item["path"]
    df_train = train_item["df"]

    for cfg in RUBERT_CONFIGS:
        reset_seed()
        print("\n" + "=" * 90)
        print(f"DATASET: {dataset_name} | MODEL: {cfg['short_name']} | TRAIN SIZE: {len(df_train)}")
        print("=" * 90)

        result = train_and_evaluate(
            df_train=df_train,
            df_test=df_test,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            num_epochs=cfg["num_epochs"],
            batch_size=cfg["batch_size"],
            name=f"[{dataset_name}] {cfg['short_name']}",
        )

        rubert_comparison_results.append({
            "dataset": dataset_name,
            "model": cfg["short_name"],
            "model_name": cfg["model_name"],
            "train_dataset": train_path.name,
            "train_size": len(df_train),
            "test_size": len(df_test),
            "num_epochs": cfg["num_epochs"],
            "batch_size": cfg["batch_size"],
            "lr": cfg["lr"],
            "balanced_accuracy": float(result["balanced_accuracy"]),
            "macro_f1": float(result["macro_f1"]),
        })

        pd.DataFrame(rubert_comparison_results).to_csv(
            RESULTS_DIR / "rubert_original_vs_stage3gpt_partial.csv",
            index=False,
        )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df_results = pd.DataFrame(rubert_comparison_results)
df_results.to_csv(RESULTS_DIR / "rubert_original_vs_stage3gpt.csv", index=False)
display(df_results)

In [ ]:
audit_path = DATA_DIR / "data_after_stage3gpt_audit.csv"
if audit_path.exists():
    audit = pd.read_csv(audit_path)
    print(f"Audit rows: {len(audit)}")
    display(audit.groupby(["source_stage", "kept"]).size().unstack(fill_value=0))
    rejected = audit[~audit["kept"]].copy()
    reason_counts = (
        rejected["reasons"]
        .dropna()
        .str.split(";")
        .explode()
        .value_counts()
        .to_frame("count")
    )
    display(reason_counts)